# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described using a Croissant schema available via a URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Explore available record sets and their fields. All entities are referenced by their `@id`.

In [ ]:
# List all available record sets by their @id
record_sets = list(dataset.record_sets.values())
print(f"Found {len(record_sets)} record sets in the dataset.")
for record_set in record_sets:
    print(f"RecordSet @id: {record_set.id}\n  Name: {getattr(record_set, 'name', None)}\n  Description: {getattr(record_set, 'description', None)}")
    # List all fields (and columns) in the record set
    if hasattr(record_set, 'fields'):
        print("  Fields:")
        for field in record_set.fields:
            print(f"    - @id: {field.id}, name: {getattr(field, 'name', None)}, type: {getattr(field, 'data_type', None)}")
    print('-' * 60)


## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis, referencing record sets and fields by their `@id`.

In [ ]:
# Extract data for each record set, referenced by @id
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set: {record_set_id}, shape: {df.shape}")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Display columns for the first available record set
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"Columns in record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No dataframes loaded.")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering numeric fields, normalizing, and grouping records by key attributes.

All field references use their `@id`.

In [ ]:
# Proceed only if there is data to analyze
if dataframes:
    df = dataframes[main_record_set_id]
    print(f"Preview DataFrame for {main_record_set_id}:")
    display(df.head())

    # Try to identify a numeric field by searching for any numeric-like columns
    numeric_cols = df.select_dtypes(include=['number', 'float', 'int']).columns.tolist()
    if not numeric_cols:
        # Try to convert columns that look like numbers
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except Exception:
                pass
        numeric_cols = df.select_dtypes(include=['number', 'float', 'int']).columns.tolist()

    if numeric_cols:
        numeric_field_id = numeric_cols[0]  # Use the first numeric column's @id
        threshold = 0  # You can adjust based on context
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to find a group field (categorical with limited cardinality)
        candidate_group_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
        group_field_id = None
        for col in candidate_group_cols:
            if df[col].nunique() > 1 and df[col].nunique() < 20:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped (mean) by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No data available for EDA.")


## 5. Visualization
Visualize the distribution of the selected numeric field, and optionally show relationships between fields using their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field_id' in locals():
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True, color='skyblue')
    plt.title(f'Distribution of Numeric Field (@id: {numeric_field_id})')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If group_field_id exists, plot a boxplot
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(8, 4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("Visualization skipped due to lack of numeric field data.")


## 6. Conclusion
In this notebook, you explored the FAIR^2 dataset following the Croissant schema, loaded record sets using `@id`, and applied typical preprocessing and visualizations using the `mlcroissant` library. Further analysis can build on these foundations to investigate predictors of knowledge adoption in Northern Kenya rangeland management.